# Supply-Chain Graph — Exploratory Data Analysis

This notebook explores the supply-chain dataset **before** we train any model.
The goal is to understand the graph structure, feature distributions, and label
patterns so we can make informed decisions during modelling.

**What we have:**
- **600 nodes** — 400 infrastructure (PORT, PLANT, WAREHOUSE, DC) + 200 context (PRODUCT, INGREDIENT)
- **1,593 directed edges** — 5 relationship types connecting the supply chain
- **Binary labels** on infrastructure nodes only: `critical` (1) vs `non-critical` (0)
- **26 engineered features** per node (one-hot categoricals + scaled numerics)

## Section 1 — Load the processed data

We load three things:
1. **`master_nodes.csv`** — the merged node table with features + labels (produced by our pipeline)
2. **`edges.csv`** — the raw edge list with relationship types
3. **`graph_data.pt`** — the serialised PyTorch Geometric `Data` object (features as tensors)

In [ ]:
import pathlib
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import torch

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

ROOT = pathlib.Path("..").resolve()
DATA_DIR = ROOT / "data" / "labelled_data"
PROCESSED_DIR = ROOT / "data" / "processed"

master = pd.read_csv(PROCESSED_DIR / "master_nodes.csv")
edges = pd.read_csv(DATA_DIR / "edges.csv")
pyg_data = torch.load(PROCESSED_DIR / "graph_data.pt", weights_only=False)

INFRA_TYPES = ["PORT", "PLANT", "WAREHOUSE", "DC"]
infra = master[master["node_type"].isin(INFRA_TYPES)].copy()

print(f"master_nodes : {master.shape}")
print(f"edges        : {edges.shape}")
print(f"PyG Data     : {pyg_data}")
print(f"\nInfra nodes  : {len(infra)}  |  Context nodes : {len(master) - len(infra)}")

## Section 2 — Label overview

Before looking at features or graph structure, we need to understand the **target variable**.

- How many nodes are `critical` vs `non-critical`?
- Does the split vary across infrastructure types?

This tells us whether the classes are balanced (they aren't — we'll see) and which
node types dominate each class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Overall label distribution (all 600 nodes) ---
label_map = {1: "critical", 0: "non-critical", -1: "unlabelled"}
overall = master["label_encoded"].map(label_map).value_counts()
colors_overall = ["#e74c3c", "#3498db", "#bdc3c7"]
overall.reindex(["critical", "non-critical", "unlabelled"]).plot.bar(
    ax=axes[0], color=colors_overall, edgecolor="black", linewidth=0.5
)
axes[0].set_title("Label distribution (all 600 nodes)")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
for bar in axes[0].patches:
    axes[0].annotate(
        f"{int(bar.get_height())}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom", fontsize=11,
    )

# --- Label counts by infrastructure node_type ---
ct = pd.crosstab(infra["node_type"], infra["label"])
ct = ct.reindex(INFRA_TYPES)
ct.plot.bar(ax=axes[1], color=["#e74c3c", "#3498db"], edgecolor="black", linewidth=0.5)
axes[1].set_title("Label counts by infrastructure type")
axes[1].set_ylabel("Count")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Label")

plt.tight_layout()
plt.show()

print("\nLabel counts by node_type:")
print(ct.to_string())

**Key observations:**
- The dataset is **imbalanced**: 240 critical vs 160 non-critical (60/40 split).
- **DCs** are overwhelmingly critical (164 of 175). This makes sense — distribution centres
  sit at the end of the supply chain, so losing one directly impacts delivery.
- **PLANTs** and **WAREHOUSEs** lean non-critical, meaning many have backup alternatives.
- **PORTs** are almost evenly split (8 critical, 7 non-critical).

## Section 3 — Feature distributions by label

Do critical and non-critical nodes look different in terms of their raw features?
If a single feature cleanly separates the two classes, the model's job is easy.
If they overlap heavily, the model will need to combine multiple features (and
graph structure) to make good predictions.

### 3a. Continuous features — histograms

In [ ]:
cont_features = ["capacity_units", "throughput_units", "recovery_days"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, cont_features):
    for label, color, lbl_name in [
        ("critical", "#e74c3c", "critical"),
        ("non-critical", "#3498db", "non-critical"),
    ]:
        subset = infra[infra["label"] == label][feat].dropna()
        ax.hist(subset, bins=25, alpha=0.6, color=color, label=lbl_name, edgecolor="white")
    ax.set_title(feat)
    ax.set_xlabel(feat)
    ax.set_ylabel("Count")
    ax.legend()

plt.suptitle("Continuous feature distributions — critical vs non-critical (infra nodes only)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

### 3b. Categorical features — stacked bar charts

How do `backup_level`, `region`, and `is_backup_capable` relate to criticality?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- backup_level ---
ct_backup = pd.crosstab(infra["backup_level"], infra["label"])
ct_backup.plot.bar(ax=axes[0], stacked=True, color=["#e74c3c", "#3498db"],
                   edgecolor="black", linewidth=0.5)
axes[0].set_title("backup_level vs label")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Label")

# --- region ---
ct_region = pd.crosstab(infra["region"], infra["label"])
ct_region.plot.bar(ax=axes[1], stacked=True, color=["#e74c3c", "#3498db"],
                   edgecolor="black", linewidth=0.5)
axes[1].set_title("region vs label")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="Label")

# --- is_backup_capable ---
ct_backup_cap = pd.crosstab(
    infra["is_backup_capable"].map({1.0: "capable", 0.0: "not capable"}),
    infra["label"],
)
ct_backup_cap.plot.bar(ax=axes[2], stacked=True, color=["#e74c3c", "#3498db"],
                       edgecolor="black", linewidth=0.5)
axes[2].set_title("is_backup_capable vs label")
axes[2].set_xlabel("")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(title="Label")

plt.suptitle("Categorical features vs label (infra nodes only)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Takeaways from feature distributions:**
- Continuous features (capacity, throughput, recovery days) show **significant overlap**
  between critical and non-critical nodes. No single numeric feature cleanly separates them.
- `backup_level = none` strongly correlates with criticality (nodes with no backup are fragile).
- `is_backup_capable = 0` (not capable) also skews toward critical — makes intuitive sense.
- Region alone doesn't look like a strong predictor; the label split is fairly even across regions.

This suggests the model will need to leverage **graph structure** (how nodes connect)
in addition to raw features to make accurate predictions — exactly what a GNN is designed to do.

## Section 4 — Graph structure visualization

A GNN learns by passing messages along edges. Understanding **how nodes connect**
is therefore just as important as understanding their features.

We'll build a NetworkX directed graph and visualize:
1. The **full 600-node graph** colored by node type
2. The **infrastructure-only subgraph** (PORT → PLANT → WAREHOUSE → DC) colored by label

In [ ]:
# Build a NetworkX DiGraph from the edge list
G = nx.DiGraph()
for _, row in master.iterrows():
    G.add_node(row["node_id"], node_type=row["node_type"],
               label=row.get("label", None))
for _, row in edges.iterrows():
    G.add_edge(row["src_id"], row["dst_id"], rel_type=row["rel_type"])

print(f"NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Is DAG (directed acyclic)? {nx.is_directed_acyclic_graph(G)}")
print(f"Weakly connected components: {nx.number_weakly_connected_components(G)}")

### 4a. Full graph — colored by node type

Each color represents a different type of node in the supply chain.
Critical infrastructure nodes are drawn with a **larger marker** so you can
spot where they sit in the overall topology.

In [ ]:
type_colors = {
    "PORT": "#e74c3c",
    "PLANT": "#2ecc71",
    "WAREHOUSE": "#f39c12",
    "DC": "#9b59b6",
    "INGREDIENT": "#1abc9c",
    "PRODUCT": "#3498db",
}

node_order = list(G.nodes())
node_types = [G.nodes[n]["node_type"] for n in node_order]
node_colors = [type_colors[t] for t in node_types]
node_labels_raw = [G.nodes[n].get("label") for n in node_order]
node_sizes = [120 if lbl == "critical" else 40 for lbl in node_labels_raw]

pos = nx.spring_layout(G, seed=42, k=0.3, iterations=80)

fig, ax = plt.subplots(figsize=(16, 12))
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.08, arrows=True,
                       arrowsize=5, edge_color="#888888")
nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=node_order,
                       node_color=node_colors, node_size=node_sizes,
                       edgecolors="black", linewidths=0.3, alpha=0.85)

legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c,
               markersize=10, label=t)
    for t, c in type_colors.items()
]
ax.legend(handles=legend_handles, loc="upper left", title="Node type", fontsize=9)
ax.set_title("Full supply-chain graph (600 nodes) — larger dots = critical", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

### 4b. Infrastructure subgraph — colored by label

This strips away PRODUCT and INGREDIENT nodes to focus on the physical
supply chain flow: **PORT → PLANT → WAREHOUSE → DC**.  Nodes are colored
by their criticality label.

In [ ]:
infra_node_ids = set(infra["node_id"])
G_infra = G.subgraph(infra_node_ids).copy()

infra_label_colors = {
    "critical": "#e74c3c",
    "non-critical": "#3498db",
}

infra_order = list(G_infra.nodes())
infra_colors = [
    infra_label_colors.get(G_infra.nodes[n].get("label"), "#cccccc")
    for n in infra_order
]
infra_shapes_map = {"PORT": "^", "PLANT": "s", "WAREHOUSE": "D", "DC": "o"}
pos_infra = nx.spring_layout(G_infra, seed=42, k=0.5, iterations=80)

fig, ax = plt.subplots(figsize=(16, 12))
nx.draw_networkx_edges(G_infra, pos_infra, ax=ax, alpha=0.12, arrows=True,
                       arrowsize=6, edge_color="#888888")

for ntype, marker in infra_shapes_map.items():
    nodes_of_type = [n for n in infra_order if G_infra.nodes[n]["node_type"] == ntype]
    colors_of_type = [
        infra_label_colors.get(G_infra.nodes[n].get("label"), "#cccccc")
        for n in nodes_of_type
    ]
    nx.draw_networkx_nodes(G_infra, pos_infra, ax=ax, nodelist=nodes_of_type,
                           node_color=colors_of_type, node_shape=marker,
                           node_size=80, edgecolors="black", linewidths=0.4, alpha=0.85)

legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#e74c3c",
               markersize=10, label="critical"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#3498db",
               markersize=10, label="non-critical"),
    plt.Line2D([], [], color="none", label=""),
    plt.Line2D([0], [0], marker="^", color="w", markerfacecolor="#999",
               markersize=10, label="PORT"),
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor="#999",
               markersize=10, label="PLANT"),
    plt.Line2D([0], [0], marker="D", color="w", markerfacecolor="#999",
               markersize=10, label="WAREHOUSE"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#999",
               markersize=10, label="DC"),
]
ax.legend(handles=legend_handles, loc="upper left", fontsize=9,
          title="Label / Type")
ax.set_title("Infrastructure subgraph (400 nodes) — colored by label", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Infrastructure subgraph: {G_infra.number_of_nodes()} nodes, "
      f"{G_infra.number_of_edges()} edges")

### 4c. Degree distribution

The **degree** of a node is the number of edges connected to it. In a directed graph
we distinguish between **in-degree** (edges coming in) and **out-degree** (edges going out).
Nodes with very high or very low degree may behave differently with respect to criticality.

In [ ]:
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())
total_deg = {n: in_deg[n] + out_deg[n] for n in G.nodes()}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(in_deg.values(), bins=30, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[0].set_title("In-degree distribution (all nodes)")
axes[0].set_xlabel("In-degree")
axes[0].set_ylabel("Count")

axes[1].hist(out_deg.values(), bins=30, color="#e67e22", edgecolor="white", alpha=0.8)
axes[1].set_title("Out-degree distribution (all nodes)")
axes[1].set_xlabel("Out-degree")

axes[2].hist(total_deg.values(), bins=30, color="#8e44ad", edgecolor="white", alpha=0.8)
axes[2].set_title("Total degree distribution (all nodes)")
axes[2].set_xlabel("Total degree")

plt.tight_layout()
plt.show()

deg_series = pd.Series(total_deg)
print(f"Degree stats — min: {deg_series.min()}, max: {deg_series.max()}, "
      f"mean: {deg_series.mean():.1f}, median: {deg_series.median():.0f}")

## Section 5 — Structural patterns and criticality

Now the key question: **does a node's position in the graph predict whether it's critical?**

If critical nodes tend to have lower degree (fewer connections = fewer alternatives = more fragile),
that's a structural signal the GNN can exploit.

In [ ]:
# Attach degree info to the infra DataFrame
infra = infra.copy()
infra["in_degree"] = infra["node_id"].map(in_deg)
infra["out_degree"] = infra["node_id"].map(out_deg)
infra["total_degree"] = infra["node_id"].map(total_deg)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 5a. Scatter: in-degree vs out-degree, colored by label ---
for label, color in [("critical", "#e74c3c"), ("non-critical", "#3498db")]:
    subset = infra[infra["label"] == label]
    axes[0].scatter(subset["in_degree"], subset["out_degree"],
                    c=color, label=label, alpha=0.6, edgecolors="black",
                    linewidth=0.3, s=40)
axes[0].set_xlabel("In-degree")
axes[0].set_ylabel("Out-degree")
axes[0].set_title("In-degree vs Out-degree (infra nodes)")
axes[0].legend()

# --- 5b. Box plot: total degree by label ---
sns.boxplot(data=infra, x="label", y="total_degree", ax=axes[1],
            palette={"critical": "#e74c3c", "non-critical": "#3498db"},
            order=["critical", "non-critical"])
axes[1].set_title("Total degree by label")
axes[1].set_xlabel("")
axes[1].set_ylabel("Total degree")

# --- 5c. Bar chart: mean degree by node_type and label ---
mean_deg = (
    infra.groupby(["node_type", "label"])["total_degree"]
    .mean()
    .unstack("label")
    .reindex(INFRA_TYPES)
)
mean_deg.plot.bar(ax=axes[2], color=["#e74c3c", "#3498db"],
                  edgecolor="black", linewidth=0.5)
axes[2].set_title("Mean total degree by type & label")
axes[2].set_xlabel("")
axes[2].set_ylabel("Mean total degree")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(title="Label")

plt.suptitle("Graph structure vs criticality", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Takeaways from structural patterns:**
- The **scatter plot** shows that critical nodes (red) cluster in different
  in-degree / out-degree regions than non-critical nodes (blue), but there is overlap.
- The **box plot** reveals whether critical nodes have systematically different total degree.
  If critical nodes tend to have *lower* degree, that signals they are bottlenecks with
  fewer alternative paths — exactly the kind of structural vulnerability a GNN can detect.
- The **mean degree chart** breaks this down by infrastructure type, showing which types
  have the strongest degree-vs-label signal.

## Section 6 — Feature correlation

Finally, let's look at how the **26 engineered features** correlate with each other
and with the label, restricted to the 400 infrastructure nodes that have labels.

**Why this matters:**
- Highly correlated features are redundant — the model doesn't gain much from both.
- Features strongly correlated with the label are likely to be important predictors.
- Features with near-zero correlation to the label will rely on graph-structure
  interactions (message passing) to contribute.

In [ ]:
import json

feature_cols_path = ROOT / "outputs" / "feature_columns.json"
with open(feature_cols_path) as f:
    feature_cols = json.load(f)

# Extract the feature matrix for infrastructure nodes only
infra_indices = master[master["node_type"].isin(INFRA_TYPES)].index.tolist()
x_infra = pyg_data.x[infra_indices].numpy()
y_infra = pyg_data.y[infra_indices].numpy()

# Build a DataFrame with feature names + the label
corr_df = pd.DataFrame(x_infra, columns=feature_cols)
corr_df["label"] = y_infra

# Full correlation matrix
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(
    corr_matrix,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.3,
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
    annot=False,
)
ax.set_title("Feature correlation heatmap (400 infrastructure nodes + label)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Show which features correlate most (and least) with the label
label_corr = corr_matrix["label"].drop("label").sort_values(ascending=False)

print("Top 10 features POSITIVELY correlated with criticality (label=1):\n")
for feat, val in label_corr.head(10).items():
    print(f"  {val:+.3f}  {feat}")

print("\nTop 10 features NEGATIVELY correlated with criticality:\n")
for feat, val in label_corr.tail(10).items():
    print(f"  {val:+.3f}  {feat}")

In [ ]:
# Horizontal bar chart of label correlations for all features
fig, ax = plt.subplots(figsize=(10, 10))
label_corr_sorted = label_corr.sort_values()
colors = ["#e74c3c" if v > 0 else "#3498db" for v in label_corr_sorted.values]
label_corr_sorted.plot.barh(ax=ax, color=colors, edgecolor="black", linewidth=0.3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson correlation with label")
ax.set_title("Feature correlation with criticality label", fontsize=13)
plt.tight_layout()
plt.show()

**Takeaways from feature correlation:**
- The `node_type_DC` one-hot feature will likely have the strongest positive correlation
  with criticality, confirming what we saw in Section 2 (164 of 175 DCs are critical).
- `backup_level_none` and `is_backup_capable = 0` should also correlate positively with
  criticality (no backup = more fragile).
- `node_type_WAREHOUSE` and `node_type_PLANT` likely correlate *negatively* (most are non-critical).
- Continuous features (`capacity_units`, `throughput_units`, `recovery_days`) probably show
  weaker correlations, meaning the GNN's ability to aggregate neighbourhood information
  will be key for accurate predictions.

---

**Summary:** The data has a clear class imbalance (60/40), strong node-type signal,
moderate structural signal from degree, and weak signal from raw numeric features.
A GNN should be well-suited because it can combine node features with multi-hop
neighbourhood structure to learn which nodes are critical bottlenecks.